# 🔍 Veri Madenciliği — Birliktelik Analizi
## 4. Ders Uygulamalı Notebook
**Haydar Kılıç | Yapay Zeka Mühendisliği**

---

Bu notebook, birliktelik analizi dersindeki tüm teorik kavramları Python ile uygulamalı olarak göstermektedir.

### 📋 İçerik
1. Temel Kavramlar (Destek, Güven, Lift)
2. Market Sepeti Verisi
3. Apriori Algoritması (sıfırdan)
4. Kural Üretimi
5. Maksimal ve Kapalı Kümeler
6. FP-Growth (sıfırdan)
7. Örüntü Değerlendirme Ölçütleri (Lift, PS, IS)
8. Simpson Paradoksu
9. Kategorik & Sürekli Özellikler
10. Sekans Örüntüleri (GSP)
11. Seyrek & Negatif Örüntüler

In [ ]:
# ============================================================
# 📦 GEREKLİ KÜTÜPHANELERİ YÜKLE
# ============================================================
import itertools
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Grafik ayarları
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Tüm kütüphaneler başarıyla yüklendi!')

---
## 1️⃣ Temel Kavramlar

### Market Sepeti Verisi

| TID | Öğeler |
|-----|--------|
| 1   | {Ekmek, Süt} |
| 2   | {Ekmek, Bez, Bira, Yumurta} |
| 3   | {Süt, Bez, Bira, Kola} |
| 4   | {Ekmek, Süt, Bez, Bira} |
| 5   | {Ekmek, Süt, Bez, Kola} |

Her işlem (transaction) bir müşterinin alışveriş sepetini temsil eder.

In [ ]:
# ============================================================
# 📊 DERS VERİSİ: Market Sepeti
# ============================================================

transactions = [
    ['Ekmek', 'Süt'],
    ['Ekmek', 'Bez', 'Bira', 'Yumurta'],
    ['Süt', 'Bez', 'Bira', 'Kola'],
    ['Ekmek', 'Süt', 'Bez', 'Bira'],
    ['Ekmek', 'Süt', 'Bez', 'Kola']
]

# İkili (0/1) gösterim oluştur
all_items = sorted(set(item for t in transactions for item in t))
binary_data = []
for t in transactions:
    row = {item: (1 if item in t else 0) for item in all_items}
    binary_data.append(row)

df_binary = pd.DataFrame(binary_data, index=[f'T{i+1}' for i in range(len(transactions))])

print('📋 İşlem Tablosu (İkili Gösterim):')
print('='*55)
print(df_binary.to_string())
print(f'\n📌 Toplam işlem sayısı (N): {len(transactions)}')
print(f'📌 Toplam öğe sayısı (d): {len(all_items)}')
print(f'📌 Öğeler: {all_items}')

In [ ]:
# ============================================================
# 📐 TEMEL FORMÜLLER: Destek Sayısı, Destek, Güven
# ============================================================

def sigma(itemset, transactions):
    """
    Destek Sayısı σ(X):
    X öğe kümesini içeren işlem sayısı.
    σ(X) = |{ tᵢ | X ⊆ tᵢ, tᵢ ∈ T }|
    """
    count = 0
    for t in transactions:
        if set(itemset).issubset(set(t)):
            count += 1
    return count

def support(itemset, transactions):
    """
    Destek s(X): X'in işlemlerde görülme ORANI.
    s(X) = σ(X) / N
    """
    return sigma(itemset, transactions) / len(transactions)

def confidence(antecedent, consequent, transactions):
    """
    Güven c(X → Y): X olduğunda Y'nin tahmin güvenilirliği.
    c(X → Y) = σ(X ∪ Y) / σ(X)
    """
    union = list(antecedent) + list(consequent)
    return sigma(union, transactions) / sigma(antecedent, transactions)

# --- Ders örneği: {Süt, Bez} → {Bira} ---
ant = ['Süt', 'Bez']
con = ['Bira']

sig_union = sigma(ant + con, transactions)
sig_ant   = sigma(ant, transactions)
N         = len(transactions)

s = sig_union / N
c = sig_union / sig_ant

print('📌 DERS ÖRNEĞİ: Kural {Süt, Bez} → {Bira}')
print('='*45)
print(f'  σ({{Süt, Bez, Bira}}) = {sig_union}')
print(f'  σ({{Süt, Bez}})       = {sig_ant}')
print(f'  N                    = {N}')
print(f'  Destek  s = {sig_union}/{N} = {s:.2f} (%{s*100:.0f})')
print(f'  Güven   c = {sig_union}/{sig_ant} = {c:.2f} (%{c*100:.0f})')
print()

# Tüm tekil öğelerin destek değerleri
print('📊 Tüm Tekil Öğelerin Destek Değerleri:')
print('-'*35)
for item in all_items:
    s_val = support([item], transactions)
    sig_val = sigma([item], transactions)
    print(f'  {item:<10} σ={sig_val}  s={s_val:.2f} (%{s_val*100:.0f})')

In [ ]:
# ============================================================
# 📊 GÖRSEL: Öğe Frekansları
# ============================================================

item_counts = {item: sigma([item], transactions) for item in all_items}
item_supports = {item: support([item], transactions) for item in all_items}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Sol: Destek sayısı
colors = ['#2ecc71' if v >= 3 else '#e74c3c' for v in item_counts.values()]
bars = ax1.bar(item_counts.keys(), item_counts.values(), color=colors, edgecolor='black', linewidth=0.8)
ax1.axhline(y=3, color='navy', linestyle='--', linewidth=2, label='minsup sayısı = 3')
ax1.set_title('Öğe Destek Sayıları (σ)', fontweight='bold')
ax1.set_ylabel('Destek Sayısı (σ)')
ax1.set_xlabel('Öğe')
ax1.legend()
for bar, val in zip(bars, item_counts.values()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, str(val),
             ha='center', va='bottom', fontweight='bold')

# Sağ: Destek oranı
colors2 = ['#2ecc71' if v >= 0.6 else '#e74c3c' for v in item_supports.values()]
bars2 = ax2.bar(item_supports.keys(), item_supports.values(), color=colors2, edgecolor='black', linewidth=0.8)
ax2.axhline(y=0.6, color='navy', linestyle='--', linewidth=2, label='minsup = %60')
ax2.set_title('Öğe Destek Oranları (s)', fontweight='bold')
ax2.set_ylabel('Destek (s)')
ax2.set_xlabel('Öğe')
ax2.set_ylim(0, 1.1)
ax2.legend()
for bar, val in zip(bars2, item_supports.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.0%}',
             ha='center', va='bottom', fontweight='bold')

green_patch = mpatches.Patch(color='#2ecc71', label='Sık (≥ minsup)')
red_patch = mpatches.Patch(color='#e74c3c', label='Seyrek (< minsup)')
fig.legend(handles=[green_patch, red_patch], loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Market Sepeti: Öğe Frekans Analizi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ minsup=%60 için SIK öğeler: Ekmek, Süt, Bez, Bira')
print('❌ ELİNEN öğeler: Kola (σ=2), Yumurta (σ=1)')

---
## 2️⃣ Apriori İlkesi ve Anti-Monotonluk

> **Teorem (Apriori İlkesi):** Bir öğe kümesi **sıksa**, tüm **alt kümeleri de sık** olmalıdır.
>
> **Tersine (Budama):** `{a,b}` sık **değilse**, `{a,b}`'yi içeren tüm üst kümeler de sık **değildir**.

Bu özelliğe **Anti-Monoton** denir:
$$X \subset Y \Rightarrow f(Y) \leq f(X)$$

In [ ]:
# ============================================================
# 🌳 APRİORİ İLKESİ: Kafes Yapısı Görselleştirme
# ============================================================

def visualize_lattice_pruning():
    """
    3 öğe (a, b, c) için öğe kafesi ve budama etkisini gösterir.
    {a,b} sık değilse, üst kümeler (abc) de budanır.
    """
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.axis('off')
    ax.set_title('Öğe Kafes Yapısı ve Apriori Budama\n(Kırmızı = budanan seyrek kümeler)', 
                 fontsize=13, fontweight='bold')

    # Düğüm pozisyonları
    nodes = {
        '∅':    (5.0, 6.8),
        'a':    (2.5, 5.2),
        'b':    (5.0, 5.2),
        'c':    (7.5, 5.2),
        'ab':   (2.0, 3.2),
        'ac':   (5.0, 3.2),
        'bc':   (8.0, 3.2),
        'abc':  (5.0, 1.2),
    }
    # Kenarlar
    edges = [
        ('∅','a'), ('∅','b'), ('∅','c'),
        ('a','ab'), ('a','ac'),
        ('b','ab'), ('b','bc'),
        ('c','ac'), ('c','bc'),
        ('ab','abc'), ('ac','abc'), ('bc','abc')
    ]
    # Sık değil olanlar (budanan)
    pruned = {'ab', 'abc'}
    frequent = {'∅','a','b','c','ac','bc'}

    for (u, v) in edges:
        x1, y1 = nodes[u]
        x2, y2 = nodes[v]
        color = '#e74c3c' if (u in pruned or v in pruned) else '#2c3e50'
        style = '--' if (u in pruned or v in pruned) else '-'
        ax.plot([x1, x2], [y1, y2], color=color, linewidth=2, linestyle=style, zorder=1)

    for node, (x, y) in nodes.items():
        if node in pruned:
            fc, ec, tc = '#e74c3c', '#c0392b', 'white'
        elif node == '∅':
            fc, ec, tc = '#95a5a6', '#7f8c8d', 'white'
        else:
            fc, ec, tc = '#2ecc71', '#27ae60', 'white'
        circle = plt.Circle((x, y), 0.5, color=fc, ec=ec, linewidth=2, zorder=2)
        ax.add_patch(circle)
        ax.text(x, y, node, ha='center', va='center', fontsize=11,
                fontweight='bold', color=tc, zorder=3)

    # Açıklama
    ax.text(1.0, 0.5, '💡 {a,b} sık değil → {abc} hiç değerlendirilmez!\n   Bu Apriori budamasıdır.',
            fontsize=10, color='#c0392b',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#fdebd0', edgecolor='#e74c3c'))
    green_p = mpatches.Patch(color='#2ecc71', label='Sık Küme')
    red_p   = mpatches.Patch(color='#e74c3c', label='Seyrek / Budanan')
    ax.legend(handles=[green_p, red_p], loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.show()

visualize_lattice_pruning()
print('Anti-Monoton Özellik: X ⊂ Y  ⟹  f(Y) ≤ f(X)')
print('Destek budaması Apriori\'nin verimli çalışmasını sağlar!')

---
## 3️⃣ Apriori Algoritması (Sıfırdan)

**Algoritma 4.1 — Sık Öğe Kümesi Üretimi:**
1. k=1 ile başla; tekil sık öğeleri bul
2. Sık (k-1)-kümelerden k-küme adayları üret (`candidate-gen`)
3. Alt kümesi seyrek olan adayları ele (`candidate-prune`)
4. Veriden destek say
5. Fk = ∅ olana dek tekrarla

In [ ]:
# ============================================================
# 🤖 APRİORİ ALGORİTMASI — TAM UYGULAMA
# ============================================================

def apriori_candidate_gen(frequent_prev):
    """
    Fk-1 × Fk-1 Birleştirme Yöntemi:
    İlk (k-2) öğesi aynı olan çiftleri birleştir.
    Leksikografik sıra gerektirir.
    """
    candidates = []
    freq_list = sorted([sorted(s) for s in frequent_prev])
    for i in range(len(freq_list)):
        for j in range(i + 1, len(freq_list)):
            A = freq_list[i]
            B = freq_list[j]
            # İlk k-2 öğe aynı mı? (Birleştirme koşulu)
            if A[:-1] == B[:-1] and A[-1] < B[-1]:
                candidates.append(A + [B[-1]])
    return [frozenset(c) for c in candidates]

def apriori_candidate_prune(candidates, frequent_prev):
    """
    candidate-prune: Herhangi bir (k-1) alt kümesi seyrek
    ise adayı listeden çıkar.
    """
    pruned = []
    freq_set = set(frequent_prev)
    for cand in candidates:
        # Tüm (k-1) boyutlu alt kümeler sık mı?
        subsets = [frozenset(s) for s in itertools.combinations(cand, len(cand)-1)]
        if all(s in freq_set for s in subsets):
            pruned.append(cand)
    return pruned

def apriori(transactions, minsup):
    """
    Tam Apriori Algoritması.
    Girdi:
        transactions: liste içinde liste (her liste = bir işlem)
        minsup: minimum destek eşiği (oran, 0-1 arası)
    Çıktı:
        Tüm sık öğe kümeleri ve destek değerleri
    """
    N = len(transactions)
    min_count = minsup * N  # kaç işlemde görünmeli

    all_items = sorted(set(item for t in transactions for item in t))
    all_frequent = {}
    iteration_log = []

    # --- Adım 1: 1-itemset ---
    F_curr = {}
    for item in all_items:
        count = sigma([item], transactions)
        if count >= min_count:
            F_curr[frozenset([item])] = count / N

    k = 1
    iteration_log.append((k, dict(F_curr)))
    all_frequent.update(F_curr)

    # --- Adım 2+: k-itemset ---
    while F_curr:
        # Aday üret
        candidates = apriori_candidate_gen(list(F_curr.keys()))
        # Budama
        candidates = apriori_candidate_prune(candidates, list(F_curr.keys()))

        F_next = {}
        for cand in candidates:
            count = sigma(list(cand), transactions)
            if count >= min_count:
                F_next[cand] = count / N

        k += 1
        if F_next:
            iteration_log.append((k, dict(F_next)))
            all_frequent.update(F_next)
        F_curr = F_next

    return all_frequent, iteration_log


# ============================================================
# ▶️  ÇALIŞTIR: minsup = %60
# ============================================================
minsup = 0.6
frequent_itemsets, log = apriori(transactions, minsup)

print(f'🔍 Apriori Algoritması — minsup = {minsup*100:.0f}% (min sayı = {minsup*len(transactions):.0f})')
print('='*60)

for k, freq in log:
    print(f'\n📦 {k}-itemset Sık Kümeler:')
    print("  {:<30} {:>15} {:>10}".format("Öğe Kümesi", "Destek Sayısı", "Destek"))
    print("  " + "-"*55)
    for itemset, sup in freq.items():
        count = int(sup * len(transactions))
        label = '{' + ', '.join(sorted(itemset)) + '}'
        status = '✅'
        print(f'  {label:<30} {count:>15}    {sup:.2f} {status}')

print(f'\n📌 Toplam sık öğe kümesi sayısı: {len(frequent_itemsets)}')

In [ ]:
# ============================================================
# 📊 GÖRSEL: Apriori — Aday vs Sık Küme Karşılaştırması
# ============================================================

# Kaba kuvvet aday sayısı vs Apriori aday sayısı
d = len(all_items)  # 6 öğe
k_values = [1, 2, 3]

brute_force_counts = []
for k in k_values:
    brute_force_counts.append(len(list(itertools.combinations(all_items, k))))

apriori_counts = [len(v) for (k, v) in log]
# Apriori total candidates per level (from log)

x = np.arange(len(k_values))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, brute_force_counts, width, label='Kaba Kuvvet (Tüm Adaylar)', 
            color='#e74c3c', alpha=0.85, edgecolor='black')
b2 = ax.bar(x + width/2, apriori_counts[:len(k_values)], width, label='Apriori (Sık Kümeler)', 
            color='#2ecc71', alpha=0.85, edgecolor='black')

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('k (itemset boyutu)')
ax.set_ylabel('Küme Sayısı')
ax.set_title('Kaba Kuvvet vs Apriori: Aday Sayısı Karşılaştırması', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{k}-itemset' for k in k_values])
ax.legend()

total_bf = sum(brute_force_counts)
total_ap = sum(apriori_counts[:len(k_values)])
reduction = (1 - total_ap/total_bf) * 100
ax.text(0.98, 0.95, f'Toplam: {total_bf} → {total_ap}\n%{reduction:.0f} azalma!',
        transform=ax.transAxes, ha='right', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))

plt.tight_layout()
plt.show()
print(f'💡 Apriori, {total_bf} adayı {total_ap} adaya düşürdü → %{reduction:.0f} tasarruf!')

---
## 4️⃣ Kural Üretimi

Her sık k-küme en fazla $2^k - 2$ kural üretir.

**Güven Anti-Monotonluğu:**  
Eğer `X → Y-X` güven eşiğini sağlamıyorsa, `X̃ → Y-X̃` (X̃ ⊂ X) de sağlamaz.

$$c = \frac{\sigma(Y)}{\sigma(X)} \quad , \quad \tilde{X} \subset X \Rightarrow \sigma(\tilde{X}) \geq \sigma(X) \Rightarrow c(\tilde{X} \to ...) \leq c(X \to ...)$$

In [ ]:
# ============================================================
# 📜 KURAL ÜRETİMİ — ap-genrules
# ============================================================

def generate_rules(frequent_itemsets, transactions, minconf):
    """
    Sık öğe kümelerinden güven eşiğini sağlayan tüm kuralları üretir.
    ap-genrules yaklaşımı:
    1. Önce 1-öğeli sağ tarafla başla
    2. Yüksek güvenli kuralları birleştirerek uzun sağ taraflar oluştur
    """
    rules = []
    N = len(transactions)

    for itemset in frequent_itemsets:
        if len(itemset) < 2:
            continue  # En az 2 öğe gerekli

        # Tüm olası sol/sağ taraf bölümlemeleri
        for size in range(1, len(itemset)):
            for consequent in itertools.combinations(itemset, size):
                consequent = frozenset(consequent)
                antecedent = itemset - consequent

                if len(antecedent) == 0:
                    continue

                sig_union = sigma(list(itemset), transactions)
                sig_ant   = sigma(list(antecedent), transactions)
                sig_con   = sigma(list(consequent), transactions)

                if sig_ant == 0:
                    continue

                sup  = sig_union / N
                conf = sig_union / sig_ant

                # Lift hesapla
                sup_con = sig_con / N
                sup_ant = sig_ant / N
                lift = sup / (sup_ant * sup_con) if (sup_ant * sup_con) > 0 else 0

                if conf >= minconf:
                    rules.append({
                        'antecedent': antecedent,
                        'consequent': consequent,
                        'support':    round(sup, 3),
                        'confidence': round(conf, 3),
                        'lift':       round(lift, 3)
                    })

    return rules


# ============================================================
# ▶️  ÇALIŞTIR: minsup=%60, minconf=%70
# ============================================================
minconf = 0.7
rules = generate_rules(frequent_itemsets, transactions, minconf)

print(f'📋 Birliktelik Kuralları (minsup={minsup*100:.0f}%, minconf={minconf*100:.0f}%)')
print('='*75)
print("  {:<38} {:>8} {:>8} {:>8}".format("Kural", "Destek", "Güven", "Lift"))
print('  ' + '-'*62)

for r in sorted(rules, key=lambda x: -x['confidence']):
    ant_str = '{' + ', '.join(sorted(r['antecedent'])) + '}'
    con_str = '{' + ', '.join(sorted(r['consequent'])) + '}'
    rule_str = f'{ant_str} → {con_str}'
    lift_icon = '⬆️' if r['lift'] > 1 else ('⬇️' if r['lift'] < 1 else '➡️')
    print(f'  {rule_str:<38} {r["support"]:>8.2f} {r["confidence"]:>8.2f} {r["lift"]:>6.2f} {lift_icon}')

print(f'\n📌 Toplam kural sayısı: {len(rules)}')
print('\n💡 Lift > 1: pozitif ilişki, Lift = 1: bağımsız, Lift < 1: negatif ilişki')

In [ ]:
# ============================================================
# 📊 GÖRSEL: Kural Dağılımı — Destek vs Güven (Lift renk)
# ============================================================

if rules:
    fig, ax = plt.subplots(figsize=(10, 6))

    supports = [r['support'] for r in rules]
    confidences = [r['confidence'] for r in rules]
    lifts = [r['lift'] for r in rules]

    sc = ax.scatter(supports, confidences, c=lifts, cmap='RdYlGn',
                    s=200, zorder=3, edgecolors='black', linewidth=0.8,
                    vmin=0.5, vmax=max(lifts) + 0.5)

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Lift Değeri', fontsize=11)

    # Kural etiketleri
    for r in rules:
        ant = '{' + ','.join(sorted(r['antecedent'])) + '}'
        con = '{' + ','.join(sorted(r['consequent'])) + '}'
        ax.annotate(f'{ant}→{con}', (r['support'], r['confidence']),
                    textcoords='offset points', xytext=(5, 5), fontsize=7.5)

    ax.axhline(minconf, color='navy', linestyle='--', linewidth=1.5, label=f'minconf={minconf*100:.0f}%')
    ax.axvline(minsup, color='darkgreen', linestyle='--', linewidth=1.5, label=f'minsup={minsup*100:.0f}%')

    ax.set_xlabel('Destek (Support)', fontsize=12)
    ax.set_ylabel('Güven (Confidence)', fontsize=12)
    ax.set_title('Birliktelik Kuralları: Destek vs Güven\n(Renk = Lift değeri)', fontweight='bold')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.1)

    plt.tight_layout()
    plt.show()
else:
    print('Belirlenen eşik değerleri için kural bulunamadı.')

---
## 5️⃣ Maksimal ve Kapalı Kümeler

| Kavram | Tanım | Özellik |
|--------|-------|----------|
| **Maksimal Sık** | Hiçbir üst kümesi sık değil | En kompakt gösterim |
| **Kapalı Sık** | Hiçbir üst kümesi aynı desteğe sahip değil | Destek korunarak minimal gösterim |

İlişki: **Maksimal ⊂ Kapalı Sık ⊂ Tüm Sık**

In [ ]:
# ============================================================
# 🔒 MAKSİMAL VE KAPALI KÜMELER
# ============================================================

def find_maximal(frequent_itemsets, transactions):
    """
    Maksimal sık kümeler: Hiçbir sık üst kümesi olmayan sık kümeler.
    """
    maximal = []
    freq_keys = list(frequent_itemsets.keys())
    for fs in freq_keys:
        is_maximal = True
        for other in freq_keys:
            if fs < other:  # fs, other'ın gerçek alt kümesi mi?
                is_maximal = False
                break
        if is_maximal:
            maximal.append(fs)
    return maximal

def find_closed(frequent_itemsets, transactions):
    """
    Kapalı sık kümeler: Hiçbir üst kümesi aynı desteğe sahip değil.
    Eğer X'in bir üst kümesi X ile aynı desteğe sahipse,
    X'i içeren her işlem o üst kümeyi de içeriyor → X gereksiz.
    """
    closed = []
    freq_keys = list(frequent_itemsets.keys())
    for fs in freq_keys:
        sup_fs = frequent_itemsets[fs]
        is_closed = True
        for other in freq_keys:
            if fs < other and abs(frequent_itemsets[other] - sup_fs) < 1e-9:
                is_closed = False
                break
        if is_closed:
            closed.append(fs)
    return closed


maximal = find_maximal(frequent_itemsets, transactions)
closed  = find_closed(frequent_itemsets, transactions)
all_freq = list(frequent_itemsets.keys())

print('📊 Sık Küme Türleri Karşılaştırması')
print('='*55)

print(f'\n🔵 TÜM sık kümeler ({len(all_freq)} adet):')
for fs in all_freq:
    label = '{' + ', '.join(sorted(fs)) + '}'
    sup = frequent_itemsets[fs]
    tags = []
    if fs in maximal: tags.append('MAKSİMAL')
    if fs in closed:  tags.append('KAPALI')
    tag_str = ' [' + ', '.join(tags) + ']' if tags else ''
    print(f'   {label:<28} s={sup:.2f}{tag_str}')

print(f'\n🟡 Kapalı sık kümeler ({len(closed)} adet):')
for fs in closed:
    label = '{' + ', '.join(sorted(fs)) + '}'
    print(f'   {label:<28} s={frequent_itemsets[fs]:.2f}')

print(f'\n🔴 Maksimal sık kümeler ({len(maximal)} adet):')
for fs in maximal:
    label = '{' + ', '.join(sorted(fs)) + '}'
    print(f'   {label:<28} s={frequent_itemsets[fs]:.2f}')

print(f'\n💡 İlişki: |Maksimal|={len(maximal)} ≤ |Kapalı|={len(closed)} ≤ |Tüm Sık|={len(all_freq)}')

In [ ]:
# ============================================================
# 📊 GÖRSEL: İç içe daire diyagramı
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('Sık Küme Hiyerarşisi', fontweight='bold', fontsize=13)

# Daireler
ellipse3 = mpatches.Ellipse((5, 3.5), 9, 5.5, fill=True,
                              facecolor='#d6eaf8', edgecolor='#2980b9', linewidth=2)
ellipse2 = mpatches.Ellipse((5, 3.5), 6.5, 3.8, fill=True,
                              facecolor='#d5f5e3', edgecolor='#27ae60', linewidth=2)
ellipse1 = mpatches.Ellipse((5, 3.5), 3.5, 2, fill=True,
                              facecolor='#fadbd8', edgecolor='#e74c3c', linewidth=2)

ax.add_patch(ellipse3)
ax.add_patch(ellipse2)
ax.add_patch(ellipse1)

ax.text(5, 3.5, f'Maksimal\n({len(maximal)} küme)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#922b21')
ax.text(5, 5.2, f'Kapalı Sık\n({len(closed)} küme)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#1e8449')
ax.text(1.5, 1.0, f'Tüm Sık Kümeler\n({len(all_freq)} küme)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#1a5276')

plt.tight_layout()
plt.show()

---
## 6️⃣ FP-Growth Algoritması

FP-Growth, veriyi **FP-ağacı** (prefix tree) adlı kompakt bir yapıda kodlar ve sık kümeleri **aday üretmeden** doğrudan çıkarır.

| | Apriori | FP-Growth |
|--|---------|----------|
| Veri geçişi | Çok | **2** |
| Aday üretimi | Var | **Yok** |
| Yoğun veri | Yavaş | **Hızlı** |

In [ ]:
# ============================================================
# 🌲 FP-GROWTH AĞACI (FP-Tree) — SIFIRDAN UYGULAMA
# ============================================================

class FPNode:
    """FP-Ağacındaki tek bir düğüm."""
    def __init__(self, item, count=0, parent=None):
        self.item   = item      # Öğe etiketi
        self.count  = count     # Sayaç
        self.parent = parent    # Ebeveyn düğüm
        self.children = {}      # Çocuk düğümler {item: FPNode}
        self.link   = None      # Aynı öğeye sahip sonraki düğüm (header table linki)

    def increment(self, count=1):
        self.count += count


class FPTree:
    """FP-Growth için Sık Örüntü Ağacı."""

    def __init__(self, transactions, minsup):
        self.minsup = minsup
        self.N = len(transactions)
        self.root = FPNode(None, 0)
        self.header = {}  # {item: (count, first_node)}
        self._build(transactions)

    def _build(self, transactions):
        """1. Geçiş: frekansları say, 2. Geçiş: ağacı inşa et."""
        # 1. Geçiş: Tekli öğe frekansları
        item_counts = Counter(item for t in transactions for item in t)
        min_count = self.minsup * self.N

        # Sık olmayan öğeleri çıkar, azalan sıraya göre sırala
        self.freq_items = {item: cnt for item, cnt in item_counts.items()
                           if cnt >= min_count}
        self.order = {item: -cnt for item, cnt in self.freq_items.items()}

        # 2. Geçiş: Her işlemi ağaca ekle
        for transaction in transactions:
            # Sık olmayan öğeleri at, azalan sıraya göre sırala
            filtered = [item for item in transaction if item in self.freq_items]
            filtered.sort(key=lambda x: self.order[x])
            if filtered:
                self._insert_transaction(filtered)

    def _insert_transaction(self, items):
        """Bir işlemi (sıralı öğeler) ağaca ekle."""
        node = self.root
        for item in items:
            if item in node.children:
                # Var olan düğümün sayacını artır
                node.children[item].increment()
            else:
                # Yeni düğüm oluştur
                new_node = FPNode(item, 1, parent=node)
                node.children[item] = new_node
                # Header table'a bağla
                self._update_header(item, new_node)
            node = node.children[item]

    def _update_header(self, item, node):
        """Header tablosunu güncelle (linked list)."""
        if item not in self.header:
            self.header[item] = [self.freq_items[item], node]
        else:
            # Zincirin sonuna bağla
            current = self.header[item][1]
            while current.link is not None:
                current = current.link
            current.link = node

    def print_tree(self, node=None, indent=0, prefix=''):
        """Ağacı metin formatında yazdır."""
        if node is None:
            node = self.root
            print('NULL (kök)')
        for child_item, child_node in sorted(node.children.items()):
            print(' ' * indent + prefix + f'{child_item}:{child_node.count}')
            self.print_tree(child_node, indent + 4, '|-- ')


# ============================================================
# ▶️  DERSTE KULLANILAN VERİ (FP-Tree Örneği)
# ============================================================
fp_transactions = [
    ['Ekmek', 'Süt'],
    ['Ekmek', 'BebekBezi', 'Bira', 'Yumurta'],
    ['Süt', 'BebekBezi', 'Bira', 'Kola'],
    ['Ekmek', 'Süt', 'BebekBezi', 'Bira'],
    ['Ekmek', 'Süt', 'BebekBezi', 'Kola']
]

fp_tree = FPTree(fp_transactions, minsup=0.6)

print('🌲 FP-Ağacı Yapısı (minsup=%60):')
print('='*40)
fp_tree.print_tree()

print('\n📋 Header Tablosu (sık öğeler):')
print("  {:<14} {:>10}".format("Öğe", "Frekans"))
print('  ' + '-'*24)
for item, (count, _) in sorted(fp_tree.header.items(),
                                key=lambda x: -x[1][0]):
    print(f'  {item:<14} {count:>10}')

In [ ]:
# ============================================================
# 🌲 FP-GROWTH: Koşullu FP-Ağaçları ile Sık Küme Çıkarımı
# ============================================================

def get_prefix_paths(tree, item):
    """
    Belirli bir öğe için önek yollarını (prefix paths) bul.
    Bu yollar koşullu FP-ağacını inşa etmek için kullanılır.
    """
    paths = []
    if item not in tree.header:
        return paths
    node = tree.header[item][1]
    while node is not None:
        path = []
        parent = node.parent
        while parent and parent.item is not None:
            path.append(parent.item)
            parent = parent.parent
        if path:
            paths.append((list(reversed(path)), node.count))
        node = node.link
    return paths

def fp_growth(transactions, minsup):
    """
    FP-Growth: Böl ve Fethet stratejisiyle sık küme çıkarımı.
    Önce en az sık öğeyle başlar, koşullu ağaçlar oluşturur.
    """
    tree = FPTree(transactions, minsup)
    min_count = minsup * len(transactions)
    frequent = {}

    def mine(tree, prefix):
        # Header tablosunu artan frekans sırasına göre işle
        items = sorted(tree.header.keys(), key=lambda x: tree.header[x][0])
        for item in items:
            new_prefix = prefix | frozenset([item])
            support_count = tree.header[item][0]
            frequent[new_prefix] = support_count / len(transactions)

            # Koşullu örüntü tabanını oluştur
            prefix_paths = get_prefix_paths(tree, item)
            if not prefix_paths:
                continue

            # Koşullu işlemleri oluştur
            cond_transactions = []
            for path, count in prefix_paths:
                cond_transactions.extend([path] * count)

            if cond_transactions:
                cond_tree = FPTree(cond_transactions, minsup)
                if cond_tree.header:  # Koşullu ağaç boş değilse devam et
                    mine(cond_tree, new_prefix)

    mine(tree, frozenset())
    return frequent


# ============================================================
# ▶️  ÇALIŞTIR ve KARŞILAŞTIR
# ============================================================
fp_results = fp_growth(fp_transactions, minsup=0.6)

print('🌲 FP-Growth Sonuçları (minsup=%60):')
print('='*50)
print("  {:<35} {:>8}".format("Öğe Kümesi", "Destek"))
print('  ' + '-'*43)
for fs, sup in sorted(fp_results.items(), key=lambda x: (len(x[0]), -x[1])):
    label = '{' + ', '.join(sorted(fs)) + '}'
    print(f'  {label:<35} {sup:.2f}')

print(f'\n📌 Toplam sık küme: {len(fp_results)}')

# Apriori ile karşılaştır
apriori_results, _ = apriori(fp_transactions, minsup=0.6)
print(f'📌 Apriori sonuç (aynı veri): {len(apriori_results)} küme')

# Küme eşleşme kontrolü
fp_sets = set(fp_results.keys())
ap_sets = set(apriori_results.keys())
if fp_sets == ap_sets:
    print('\n✅ FP-Growth ve Apriori AYNI sonuçları verdi!')
else:
    diff = fp_sets.symmetric_difference(ap_sets)
    print(f'⚠️ Fark var: {diff}')

---
## 7️⃣ Örüntü Değerlendirme Ölçütleri

Güven ölçütü tek başına yeterli değildir! `s(Y)`'yi dikkate almaz.

| Ölçüt | Formül | Yorum |
|-------|--------|-------|
| **Lift** | $\frac{s(A,B)}{s(A) \cdot s(B)}$ | >1: pozitif, =1: bağımsız, <1: negatif |
| **PS** | $s(A,B) - s(A) \cdot s(B)$ | 0: bağımsız |
| **ϕ-korelasyon** | Normalize PS | [-1, +1] |
| **IS (Cosine)** | $\frac{s(A,B)}{\sqrt{s(A) \cdot s(B)}}$ | [0, 1] |

In [ ]:
# ============================================================
# 📏 OBJEKTİF İLGİ ÖLÇÜTLERİ
# ============================================================

def compute_measures(A, B, transactions):
    """
    A ve B öğe kümeleri için tüm ilgi ölçütlerini hesapla.
    """
    N = len(transactions)
    s_A   = sigma(A, transactions) / N
    s_B   = sigma(B, transactions) / N
    s_AB  = sigma(A + B, transactions) / N

    # Güven
    conf_AB = s_AB / s_A if s_A > 0 else 0

    # Lift (İlgi Faktörü)
    lift = s_AB / (s_A * s_B) if (s_A * s_B) > 0 else 0

    # PS (Piatetsky-Shapiro)
    ps = s_AB - s_A * s_B

    # φ-Korelasyonu
    denom_phi = (s_A * (1 - s_A) * s_B * (1 - s_B)) ** 0.5
    phi = (s_AB - s_A * s_B) / denom_phi if denom_phi > 0 else 0

    # IS (Cosine / Jaccard-benzeri)
    is_val = s_AB / (s_A * s_B) ** 0.5 if (s_A * s_B) > 0 else 0

    return {
        's(A)': s_A, 's(B)': s_B, 's(A,B)': s_AB,
        'Güven': conf_AB, 'Lift': lift,
        'PS': ps, 'ϕ': phi, 'IS': is_val
    }


# ============================================================
# ÖRNEK 1 — Sahte Pozitif (Çay → Kahve)
# Güven %75 ama kahve zaten %80 sık!
# ============================================================
print('📌 ÖRNEK 1: Çay → Kahve (Sahte Pozitif)')
print('Contingency tablosu:')

# f11=150, f10=50, f01=650, f00=150  (Çay + Kahve durumu)
N_ex = 1000
f11 = 150  # Çay ve Kahve birlikte
f10 = 50   # Çay var, Kahve yok
f01 = 650  # Çay yok, Kahve var
f00 = 150  # İkisi de yok

s_A_ex  = (f11 + f10) / N_ex  # s(Çay)
s_B_ex  = (f11 + f01) / N_ex  # s(Kahve)
s_AB_ex = f11 / N_ex

conf_ex = s_AB_ex / s_A_ex
lift_ex = s_AB_ex / (s_A_ex * s_B_ex)
ps_ex   = s_AB_ex - s_A_ex * s_B_ex

data_table = pd.DataFrame({'Kahve (1)': [f11, f01, f11+f01],
                            'Kahve (0)': [f10, f00, f10+f00],
                            'Toplam': [f11+f10, f01+f00, N_ex]},
                           index=['Çay (1)', 'Çay (0)', 'Toplam'])
print(data_table.to_string())
print(f'\n  s(Çay)   = {s_A_ex:.2f}  s(Kahve) = {s_B_ex:.2f}  s(Çay,Kahve) = {s_AB_ex:.2f}')
print(f'  Güven    = {conf_ex:.2f}  ← Yüksek görünüyor!')
print(f'  Lift     = {lift_ex:.2f}  ← 1 altı → NEGATİF ilişki!')
print(f'  PS       = {ps_ex:.3f} ← Negatif')
print(f'  ⚠️  Kahve zaten %{s_B_ex*100:.0f} sıklıkla alınıyor. Çay içmek kahve almayı AZALTIYOR!')

print()

# ÖRNEK 2 — Sahte Negatif (Çay → Bal)
print('📌 ÖRNEK 2: Çay → Bal (Sahte Negatif)')
f11b = 100;  f10b = 100;  f01b = 20;  f00b = 780
s_A_b  = (f11b + f10b) / N_ex
s_B_b  = (f11b + f01b) / N_ex
s_AB_b = f11b / N_ex
conf_b = s_AB_b / s_A_b
lift_b = s_AB_b / (s_A_b * s_B_b)

print(f'  s(Çay) = {s_A_b:.2f}  s(Bal) = {s_B_b:.2f}  s(Çay,Bal) = {s_AB_b:.2f}')
print(f'  Güven  = {conf_b:.2f}  ← Sadece %50, minconf geçemiyor!')
print(f'  Lift   = {lift_b:.2f}  ← 1 üstü → POZİTİF ilişki (güçlü!)')
print(f'  ✅ Çay içmek bal kullanımını %{s_B_b*100:.0f} beklentiden çok artırıyor!')

In [ ]:
# ============================================================
# 📊 GÖRSEL: Lift Yorumlama
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

scenarios = [
    {'title': 'Negatif İlişki\nÇay → Kahve', 'lift': lift_ex, 'conf': conf_ex,
     'color': '#e74c3c', 'note': f'Lift={lift_ex:.2f} < 1\nGüven yanıltıcı!'},
    {'title': 'Bağımsız\n(Örnek)', 'lift': 1.0, 'conf': 0.6,
     'color': '#95a5a6', 'note': 'Lift=1.0\nBağımsız'},
    {'title': 'Pozitif İlişki\nÇay → Bal', 'lift': lift_b, 'conf': conf_b,
     'color': '#2ecc71', 'note': f'Lift={lift_b:.2f} > 1\nGüçlü ilişki!'}
]

for ax, sc in zip(axes, scenarios):
    bars = ax.bar(['Güven', 'Lift'], [sc['conf'], sc['lift']],
                  color=[sc['color'], sc['color']], alpha=0.8, edgecolor='black')
    ax.axhline(1, color='black', linestyle='--', linewidth=1.5, label='Bağımsızlık çizgisi (Lift=1)')
    ax.set_title(sc['title'], fontweight='bold')
    ax.set_ylim(0, max(4, sc['lift'] + 0.5))
    for bar, val in zip(bars, [sc['conf'], sc['lift']]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', fontweight='bold')
    ax.text(0.5, 0.95, sc['note'], transform=ax.transAxes,
            ha='center', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    ax.legend(fontsize=8)

plt.suptitle('Güven vs Lift: Yanıltıcı Durumlar', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8️⃣ Simpson Paradoksu

> Gizli (karıştırıcı) değişkenler nedeniyle, iki değişken arasındaki ilişki **birleştirilmiş veride** bir yönde görünürken, **katmanlı veride** farklı — hatta ters yönde görünebilir.

In [ ]:
# ============================================================
# 🤯 SİMPSON PARADOKSU — HDTV & Egzersiz Aleti Örneği
# ============================================================

# Ders verileri: HDTV ile Egzersiz Aleti satın alma
data = {
    'Öğrenci': {
        'HDTV Aldı':    {'Egzersiz Aldı': 10,  'Egzersiz Almadı': 90},   # 10%
        'HDTV Almadı':  {'Egzersiz Aldı': 50,  'Egzersiz Almadı': 373},  # ~11.8%
    },
    'Çalışan': {
        'HDTV Aldı':    {'Egzersiz Aldı': 200, 'Egzersiz Almadı': 146},  # ~57.7%
        'HDTV Almadı':  {'Egzersiz Aldı': 60,  'Egzersiz Almadı': 44},   # ~57.7% ≈ 58.1%
    }
}

print('🔍 Simpson Paradoksu Analizi: HDTV → Egzersiz Aleti')
print('='*60)

total_hdtv_exercise  = 0
total_hdtv           = 0
total_nhdtv_exercise = 0
total_nhdtv          = 0

for group, d in data.items():
    h_e  = d['HDTV Aldı']['Egzersiz Aldı']
    h_ne = d['HDTV Aldı']['Egzersiz Almadı']
    nh_e  = d['HDTV Almadı']['Egzersiz Aldı']
    nh_ne = d['HDTV Almadı']['Egzersiz Almadı']

    h_total  = h_e + h_ne
    nh_total = nh_e + nh_ne

    r_hdtv  = h_e / h_total
    r_nhdtv = nh_e / nh_total

    total_hdtv_exercise  += h_e
    total_hdtv           += h_total
    total_nhdtv_exercise += nh_e
    total_nhdtv          += nh_total

    comp = '⬆️  (HDTV → daha çok egzersiz)' if r_hdtv > r_nhdtv else '⬇️  (HDTV → daha az egzersiz)'
    print(f'\n  👥 Grup: {group}')
    print(f'     HDTV Alanlar    : {h_e}/{h_total}  = %{r_hdtv*100:.1f}')
    print(f'     HDTV Almayanlar : {nh_e}/{nh_total} = %{r_nhdtv*100:.1f}')
    print(f'     → {comp}')

# Birleştirilmiş veri
r_combined_hdtv  = total_hdtv_exercise / total_hdtv
r_combined_nhdtv = total_nhdtv_exercise / total_nhdtv

print(f'\n  📊 BİRLEŞTİRİLMİŞ VERİ (Tüm gruplar):')
print(f'     HDTV Alanlar    : {total_hdtv_exercise}/{total_hdtv}  = %{r_combined_hdtv*100:.1f}')
print(f'     HDTV Almayanlar : {total_nhdtv_exercise}/{total_nhdtv} = %{r_combined_nhdtv*100:.1f}')
if r_combined_hdtv > r_combined_nhdtv:
    print(f'     → ⬆️  BİRLEŞİK: HDTV alanlar daha çok egzersiz alıyor!')

print()
print('🚨 PARADOKS!')
print('   Her grupta HDTV alanlar DAHA AZ egzersiz alıyor.')
print('   Ama birleşik veride tam TERSİ görünüyor!')
print('   Neden? Çalışanlar hem HDTV hem egzersiz aleti satın alma eğiliminde.')
print('\n💡 Çözüm: Gizli değişkeni (müşteri tipi) belirle ve veriyi katmanla!')

In [ ]:
# ============================================================
# 📊 GÖRSEL: Simpson Paradoksu
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

groups = ['Öğrenci', 'Çalışan', 'Birleşik']
hdtv_rates = [10/100, 200/346, r_combined_hdtv]
nhdtv_rates = [50/423, 60/104, r_combined_nhdtv]
colors_hdtv  = ['#3498db'] * 3
colors_nhdtv = ['#e67e22'] * 3

for i, (ax, grp, r_h, r_n) in enumerate(zip(axes, groups, hdtv_rates, nhdtv_rates)):
    bars = ax.bar(['HDTV Aldı', 'HDTV Almadı'], [r_h, r_n],
                  color=[colors_hdtv[i], colors_nhdtv[i]],
                  edgecolor='black', alpha=0.85)
    for bar, val in zip(bars, [r_h, r_n]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'%{val*100:.1f}', ha='center', fontweight='bold', fontsize=11)

    ax.set_title(f'Grup: {grp}', fontweight='bold')
    ax.set_ylabel('Egzersiz Aleti Alma Oranı')
    ax.set_ylim(0, 0.85)

    if i < 2:  # Katmanlı
        direction = '⬇️ HDTV daha az' if r_h < r_n else '⬆️ HDTV daha fazla'
        color_note = '#e74c3c' if r_h < r_n else '#27ae60'
    else:  # Birleşik
        direction = '⬆️ BİRLEŞİK: HDTV daha fazla!'
        color_note = '#e74c3c'
    ax.text(0.5, 0.95, direction, transform=ax.transAxes,
            ha='center', va='top', fontsize=10, color=color_note,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=color_note))

plt.suptitle('Simpson Paradoksu: Katmanlı vs Birleştirilmiş Veri\n'
             'Her grupta HDTV daha az egzersiz → Ama birleşik veride tam tersi!',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9️⃣ Kategorik & Sürekli Özellikler

Gerçek veriler her zaman ikili değildir. Kategorik ve sürekli özellikler için dönüşüm gerekir.

In [ ]:
# ============================================================
# 🏷️ KATEGORİK ÖZELLİKLER: İkili Forma Dönüşüm
# ============================================================

# İnternet anketi verisi
internet_data = pd.DataFrame({
    'ID':           [1, 2, 3, 4, 5, 6, 7, 8],
    'Cinsiyet':     ['Erkek','Kadın','Erkek','Kadın','Erkek','Kadın','Erkek','Kadın'],
    'EgitimDuzeyi': ['Lisans','Lisansüstü','Lise','Lisans','Lisansüstü','Lise','Lisans','Lisans'],
    'EvBilgisayari':['Evet','Evet','Evet','Hayır','Evet','Evet','Hayır','Evet'],
    'OnlineAlisveris':['Evet','Hayır','Evet','Evet','Hayır','Evet','Evet','Hayır'],
    'GizlilikKaygisi':['Evet','Evet','Hayır','Evet','Evet','Hayır','Evet','Hayır']
})

print('📋 Orijinal Anket Verisi:')
print(internet_data.to_string(index=False))

# --- İkili forma dönüşüm ---
# Her özellik-değer çifti → yeni ikili sütun
binary_cols = {}

# Kategorik sütunlar: her değer için ayrı sütun
for col in ['Cinsiyet', 'EgitimDuzeyi']:
    for val in internet_data[col].unique():
        binary_cols[f'{col}={val}'] = (internet_data[col] == val).astype(int)

# İkili sütunlar: sadece 'Evet'i al
for col in ['EvBilgisayari', 'OnlineAlisveris', 'GizlilikKaygisi']:
    binary_cols[f'{col}=Evet'] = (internet_data[col] == 'Evet').astype(int)

df_bin = pd.DataFrame(binary_cols)
df_bin.index = internet_data['ID']
df_bin.index.name = 'ID'

print('\n📋 İkili (0/1) Gösterim:')
print(df_bin.to_string())
print(f'\n✅ {len(internet_data.columns)-1} özellik → {len(df_bin.columns)} ikili öğe')

In [ ]:
# ============================================================
# 📊 SÜREKLİ ÖZELLİKLER: Ayrıştırma Tabanlı Yöntem
# ============================================================

# Yaş ve chat kullanım verisi
np.random.seed(42)
n = 200

# Gençler ve yaşlılar için farklı chat eğilimi
ages_young = np.random.randint(16, 26, size=80)
ages_mid   = np.random.randint(26, 44, size=80)
ages_old   = np.random.randint(44, 64, size=40)

chat_young = np.random.choice([1, 0], size=80, p=[0.82, 0.18])  # Ders örneği: %81.5
chat_mid   = np.random.choice([1, 0], size=80, p=[0.45, 0.55])
chat_old   = np.random.choice([1, 0], size=40, p=[0.30, 0.70])  # Ders örneği: %30

ages = np.concatenate([ages_young, ages_mid, ages_old])
chat = np.concatenate([chat_young, chat_mid, chat_old])

df_age = pd.DataFrame({'Yas': ages, 'ChatKullanimi': chat})

# Ayrıştırma: 4 yıllık aralıklar
bins_4 = list(range(16, 68, 4))
df_age['Aralik_4yil'] = pd.cut(df_age['Yas'], bins=bins_4, right=False)

# Ayrıştırma: geniş aralıklar (8 yıl)
bins_8 = list(range(16, 68, 8))
df_age['Aralik_8yil'] = pd.cut(df_age['Yas'], bins=bins_8, right=False)

# Aralık başına chat oranı
chat_by_4 = df_age.groupby('Aralik_4yil')['ChatKullanimi'].agg(['mean', 'count'])
chat_by_8 = df_age.groupby('Aralik_8yil')['ChatKullanimi'].agg(['mean', 'count'])

# Görsel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, chat_by, title, bins in zip(
        axes,
        [chat_by_4, chat_by_8],
        ['4 Yıllık Aralıklar (Optimal)', '8 Yıllık Aralıklar (Çok Geniş)'],
        [bins_4, bins_8]):

    labels = [str(iv) for iv in chat_by.index]
    means  = chat_by['mean'].values
    counts = chat_by['count'].values

    bar_colors = ['#2ecc71' if m >= 0.7 else ('#f39c12' if m >= 0.4 else '#e74c3c')
                  for m in means]
    bars = ax.bar(range(len(labels)), means, color=bar_colors, edgecolor='black', alpha=0.85)

    ax.axhline(0.7, color='navy', linestyle='--', linewidth=2, label='minconf=%70')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Chat Kullanım Oranı')
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontweight='bold')
    ax.legend()

    for bar, val, cnt in zip(bars, means, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.0%}\n(n={cnt})', ha='center', fontsize=7.5)

plt.suptitle('Ayrıştırma Tabanlı Yöntem: Yaş Aralığı Seçiminin Etkisi\n'
             'Çok geniş aralık → Kurallar kaybolabilir!',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Aralık genişliği çok büyük olursa gruplar karışır ve anlamlı kurallar kaybolur!')
print('💡 Aralık genişliği çok küçük olursa destek düşer ve kurallar yeterince görülmez!')

---
## 🔟 Sekans Örüntüleri (GSP Algoritması)

**Sekans:** $s = \langle e_1\ e_2\ \ldots\ e_n \rangle$, her $e_j$ bir element (işlem), içindeki her $i$ bir olaydır.

**Önemli Farklar (Itemset'ten):**
- Sıra önemlidir: $\langle a, b \rangle \neq \langle b, a \rangle$
- Olaylar farklı elementlerde tekrar edebilir
- Apriori ilkesi hâlâ geçerlidir

In [ ]:
# ============================================================
# ⏱️ SEKANS ÖRÜNTÜSÜ MADENCİLİĞİ — GSP Algoritması
# ============================================================

def is_subsequence(pattern, sequence):
    """
    pattern, sequence'ın alt-sekansı mı?
    Her element için pattern'ın corresponding element'i
    sequence'ın bir sonraki elementinin alt kümesi olmalı.
    """
    seq_idx = 0
    for pat_element in pattern:
        found = False
        while seq_idx < len(sequence):
            # pat_element, sequence[seq_idx]'in alt kümesi mi?
            if set(pat_element).issubset(set(sequence[seq_idx])):
                found = True
                seq_idx += 1
                break
            seq_idx += 1
        if not found:
            return False
    return True

def seq_support(pattern, seq_database):
    """Bir sekansın veri tabanındaki destek değeri."""
    count = sum(1 for seq in seq_database if is_subsequence(pattern, seq))
    return count / len(seq_database)

def gsp(seq_database, minsup):
    """
    GSP (Generalized Sequential Patterns) Algoritması.
    Apriori mantığını sekans verilerine uygular.
    """
    all_items = sorted(set(item for seq in seq_database
                           for element in seq for item in element))
    N = len(seq_database)
    freq_sequences = {}

    # k=1: Tekli sekanslar
    F_curr = {}
    for item in all_items:
        pat = ((item,),)
        s = seq_support(pat, seq_database)
        if s >= minsup:
            F_curr[pat] = s

    freq_sequences.update(F_curr)

    # k=2: İkili sekanslar
    F_next = {}
    items = [p[0][0] for p in F_curr.keys()]
    for i in items:
        for j in items:
            # Ayrı elementlerde: <{i}{j}>
            pat1 = ((i,), (j,))
            s1 = seq_support(pat1, seq_database)
            if s1 >= minsup:
                F_next[pat1] = s1
            # Aynı elementde: <{i,j}> (i < j)
            if i < j:
                pat2 = ((i, j),)
                s2 = seq_support(pat2, seq_database)
                if s2 >= minsup:
                    F_next[pat2] = s2

    freq_sequences.update(F_next)
    return freq_sequences


# ============================================================
# ▶️  WEB ZİYARET SIRASI ÖRNEĞİ
# ============================================================
# Müşterilerin web sayfası ziyaret sıraları
seq_db = [
    [('Anasayfa',), ('Elektronik',), ('Kameralar',), ('Siparis',)],
    [('Anasayfa',), ('Giyim',), ('Elektronik',), ('Siparis',)],
    [('Elektronik',), ('Kameralar',), ('Siparis',)],
    [('Anasayfa',), ('Elektronik',), ('Siparis',)],
    [('Anasayfa',), ('Giyim',), ('Siparis',)],
]

print('📋 Web Ziyaret Sekans Veri Tabanı:')
for i, seq in enumerate(seq_db, 1):
    labels = ' → '.join([e[0] for e in seq])
    print(f'  Kullanıcı {i}: {labels}')

freq_seqs = gsp(seq_db, minsup=0.6)

print(f'\n🔍 Sık Sekanslar (minsup=%60):')
print("  {:<40} {:>8}".format("Sekans", "Destek"))
print('  ' + '-'*48)
for pat, s in sorted(freq_seqs.items(), key=lambda x: (-len(x[0]), -x[1])):
    if len(pat) == 1:
        label = f'⟨{{"{pat[0][0]}"}}⟩'
    else:
        elements = ['{' + ','.join(f'"{x}"' for x in e) + '}' for e in pat]
        label = '⟨' + ' → '.join(elements) + '⟩'
    print(f'  {label:<40} {s:.2f}')

# Alt-sekans testi
print('\n📌 Alt-Sekans Testi:')
s = [('Elektronik', 'Kameralar'), ('Siparis',), ('Kameralar',)]
test_cases = [
    ([('Elektronik',), ('Siparis',)], True),
    ([('Kameralar',), ('Siparis',)], True),
    ([('Anasayfa',), ('Siparis',), ('Elektronik',)], False),
]
seq_test = [('Elektronik', 'Bira'), ('Bez',), ('Bira',)]
s_test   = [('Elektronik',), ('Bira',)]
print(f'  s = {seq_test}')
print(f'  ⟨Elektronik⟩⟨Bira⟩ alt-sekans mı? {is_subsequence(s_test, seq_test)}')

---
## 1️⃣1️⃣ Seyrek & Negatif Örüntüler

Bazen **seyrek ama ilginç** örüntüler vardır:
- **Negatif korelasyon:** DVD ve VCR birlikte seyrek (rakip ürünler)
- **h-Güven (All-Confidence):** Çapraz-destek örüntülerini elimine eder

In [ ]:
# ============================================================
# ❌ SEYREK & NEGATİF ÖRÜNTÜLER
# ============================================================

def h_confidence(itemset, transactions):
    """
    h-Güven (All-Confidence):
    h-conf(X) = s(X) / max_i(s(i_i))
    Çapraz-destek örüntülerini elimine eden anti-monoton ölçüt.
    """
    s_X = support(list(itemset), transactions)
    max_s = max(support([item], transactions) for item in itemset)
    return s_X / max_s if max_s > 0 else 0


# Ders örneği: p sık (%83), q ve r seyrek (%17) ama q ile r daima birlikte!
print('📌 Çarpık Destek Dağılımı Problemi')
print('='*50)

# p: 83 işlemde var, q: 17, r: 17
# q ve r her zaman birlikte görünüyor
trans_skewed = []
# p ve q birlikte (sahte ilişki)
for _ in range(14):
    trans_skewed.append(['p', 'q', 'r'])   # q ve r birlikte
for _ in range(3):
    trans_skewed.append(['p', 'q'])         # p ve q
for _ in range(66):
    trans_skewed.append(['p'])              # sadece p
for _ in range(17):
    trans_skewed.append(['x', 'y'])         # diğer

N_sk = len(trans_skewed)

s_p  = support(['p'], trans_skewed)
s_q  = support(['q'], trans_skewed)
s_r  = support(['r'], trans_skewed)
s_pq = support(['p', 'q'], trans_skewed)
s_qr = support(['q', 'r'], trans_skewed)

hconf_pq = h_confidence({'p', 'q'}, trans_skewed)
hconf_qr = h_confidence({'q', 'r'}, trans_skewed)

print(f'  Destek değerleri: s(p)={s_p:.2f}, s(q)={s_q:.2f}, s(r)={s_r:.2f}')
print(f'  s({{p,q}})={s_pq:.2f}, s({{q,r}})={s_qr:.2f}')

print()
print('  Çapraz-Destek Oranı r(X) = min_support / max_support:')
r_pq = min(s_p, s_q) / max(s_p, s_q)
r_qr = min(s_q, s_r) / max(s_q, s_r)
msg_pq = "⚠️ Çapraz-destek örüntüsü!" if r_pq < 0.5 else "Normal"
print(f'  r({{p,q}}) = {r_pq:.2f}  → {msg_pq}')
msg_qr = "✅ Dengeli örüntü" if r_qr >= 0.5 else ""
print(f'  r({{q,r}}) = {r_qr:.2f}  → {msg_qr}')

print()
print('  h-Güven (All-Confidence):')
msg_hpq = "❌ Düşük: p çok sık, sahte ilişki!" if hconf_pq < 0.5 else "✅"
print(f'  h-conf({{p,q}}) = {hconf_pq:.2f}  → {msg_hpq}')
msg_hqr = "✅ Yüksek: q ve r gerçekten birlikte!" if hconf_qr >= 0.5 else ""
print(f'  h-conf({{q,r}}) = {hconf_qr:.2f}  → {msg_hqr}')
print()
print('💡 Standart minsup=%20 ile:')
print(f'   {{q,r}} sık mı? {"✅ Evet" if s_qr >= 0.2 else "❌ Hayır, kaçırılıyor!"}')
print(f'   {{p,q}} sık mı? {"✅ Evet" if s_pq >= 0.2 else "❌ Hayır"}')
print('   ⚠️  h-Güven, bu ayrımı yapar!')

In [ ]:
# ============================================================
# ❌➕ NEGATİF KORELASYONLU ÜRÜNLER (Rakip Ürünler)
# ============================================================

print('📌 Negatif Korelasyon: Rakip Ürün Analizi')
print('='*50)

# Örnek: DVD ve VCR rakip ürünler (biri alınırsa diğeri alınmaz)
competitor_trans = [
    ['DVD'],
    ['DVD', 'RemoteControl'],
    ['VCR'],
    ['VCR', 'RemoteControl'],
    ['DVD'],
    ['VCR'],
    ['DVD', 'TV'],
    ['VCR', 'TV'],
    ['DVD'],
    ['VCR'],
]

N_c = len(competitor_trans)
items_c = ['DVD', 'VCR', 'RemoteControl', 'TV']

print('  Tekil öğe destekleri:')
for item in items_c:
    s = support([item], competitor_trans)
    print(f'    s({item}) = {s:.2f}')

print('\n  Birlikte destek ve ilgi ölçütleri:')
pairs = [(['DVD', 'VCR'], 'Rakip'), (['DVD', 'RemoteControl'], 'Tamamlayıcı')]

for pair, kind in pairs:
    s_pair = support(pair, competitor_trans)
    s_a    = support([pair[0]], competitor_trans)
    s_b    = support([pair[1]], competitor_trans)
    lift_val = s_pair / (s_a * s_b) if (s_a * s_b) > 0 else 0
    ps_val   = s_pair - s_a * s_b

    neg_corr = s_pair < s_a * s_b
    icon = '⚔️ Negatif' if neg_corr else '🤝 Pozitif'

    print(f'\n  Çift: {pair} ({kind})')
    print(f'    s({pair}) = {s_pair:.2f}')
    print(f'    Lift = {lift_val:.2f}  →  {icon} korelasyon')
    print(f'    PS   = {ps_val:.3f}')
    if neg_corr:
        print(f'    → Biri alınırsa diğeri satın alınmıyor! (Rakip ürünler)')

# Görsel
fig, ax = plt.subplots(figsize=(8, 5))
comparisons = ['DVD vs VCR\n(Rakip)', 'DVD vs Remote\n(Tamamlayıcı)']
lift_vals = [
    support(['DVD', 'VCR'], competitor_trans) /
    (support(['DVD'], competitor_trans) * support(['VCR'], competitor_trans)),
    support(['DVD', 'RemoteControl'], competitor_trans) /
    (support(['DVD'], competitor_trans) * support(['RemoteControl'], competitor_trans))
]
colors_bar = ['#e74c3c' if l < 1 else '#2ecc71' for l in lift_vals]
bars = ax.bar(comparisons, lift_vals, color=colors_bar, edgecolor='black', alpha=0.85, width=0.4)
ax.axhline(1, color='navy', linestyle='--', linewidth=2, label='Bağımsızlık (Lift=1)')
for bar, val in zip(bars, lift_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'Lift={val:.2f}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('Lift Değeri')
ax.set_title('Rakip vs Tamamlayıcı Ürün İlişkileri', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 📊 Genel Özet ve Algoritma Karşılaştırması

In [ ]:
# ============================================================
# 📊 FINAL: Algoritma Karşılaştırma Tablosu
# ============================================================

comparison = pd.DataFrame({
    'Özellik': [
        'Veri Geçişi', 'Aday Üretimi', 'Bellek Kullanımı',
        'Seyrek Veri', 'Yoğun Veri', 'Uygulama Kolaylığı'
    ],
    'Apriori': [
        'Çok (k+1 geçiş)', 'Var (FₖxFₖ)', 'Orta',
        '✅ İyi', '❌ Yavaş', '✅ Kolay'
    ],
    'FP-Growth': [
        'Sadece 2', 'Yok!', 'Yapıya bağlı',
        '✅ İyi', '✅ Hızlı', '⚠️ Karmaşık'
    ],
    'GSP (Sekans)': [
        'Çok', 'Var', 'Büyük',
        '✅ İyi', '❌ Yavaş', '⚠️ Orta'
    ]
})

comparison = comparison.set_index('Özellik')
print('📋 Algoritma Karşılaştırma Tablosu')
print('='*65)
print(comparison.to_string())

print()
print('='*65)
print('📚 DERS KAPSAMININ ÖZETİ')
print('='*65)
topics = [
    ('1', 'Temel Kavramlar', 'Öğe kümesi, destek, güven, birliktelik kuralı'),
    ('2', 'Apriori İlkesi', 'Anti-monotonluk, budama stratejisi'),
    ('3', 'Apriori Algoritması', 'candidate-gen, candidate-prune, hash ağacı'),
    ('4', 'Kural Üretimi', 'ap-genrules, güven anti-monotonluğu'),
    ('5', 'Kompakt Gösterim', 'Maksimal ve kapalı sık kümeler'),
    ('6', 'FP-Growth', 'FP-ağacı, koşullu ağaçlar, aday üretmeden madencilik'),
    ('7', 'Örüntü Değerlendirme', 'Lift, PS, φ-korelasyonu, IS ölçütleri'),
    ('8', 'Simpson Paradoksu', 'Gizli değişkenler, katmanlı analiz'),
    ('9', 'Kategorik & Sürekli', 'Binarizasyon, ayrıştırma, min-Apriori'),
    ('10', 'Sekans Örüntüleri', 'GSP algoritması, zamanlama kısıtları'),
    ('11', 'Seyrek & Negatif', 'h-güven, hyperclique, rakip ürünler'),
]
for num, topic, desc in topics:
    print(f'  {num:>2}. {topic:<25} → {desc}')

print()
print('✅ Notebook tamamlandı!')

---

## 🎯 Alıştırmalar

1. `minsup = 0.4` ile Apriori algoritmasını çalıştırın. Kaç sık küme bulundu? Önceki sonuçla karşılaştırın.

2. `{Ekmek} → {Süt}` kuralının destek, güven ve lift değerlerini elle hesaplayın, ardından fonksiyonlarla doğrulayın.

3. Yeni bir işlem ekleyin: `['Ekmek', 'Süt', 'Bira', 'Bez']` — bu hangi kuralların güven değerini değiştirir?

4. FP-Tree'yi `minsup = 0.4` ile yeniden inşa edin ve ağaç yapısını Apriori sonucuyla karşılaştırın.

5. `{Bebek Bezi} → {Bira}` klasik örneği için veri yaratın ve lift değerini hesaplayın.

---
*Veri Madenciliği | Birliktelik Analizi | Yapay Zeka Mühendisliği*